In [ ]:
import pandas as pd
import numpy as np
import torch
import os
from xgboost import XGBClassifier

# --- LOCAL MODULES ---
import sys
sys.path.append('..')
from src.features import load_and_engineer_features

# --- CONFIG ---
DATA_DIR = "../data/processed"
MODEL_DIR = "../models"

# 1. Load & Engineer Data
# This ensures we have 'fuel_density', 'danger_index', etc.
X, y, feature_names = load_and_engineer_features(f"{DATA_DIR}/final_dataset.csv")

# Re-load DF to attach these engineered features properly for the app
df = pd.read_csv(f"{DATA_DIR}/final_dataset.csv")
if 'address' not in df.columns:
    df['address'] = df['id'].astype(str)

# Attach the engineered features (X is numpy, so we need to map back if possible, 
# but load_and_engineer_features returns X interaction terms. 
# Let's manually recreate the simple ones we need for logic).

# Re-implementing simple features for readability in App Data
df['estimated_lot_area'] = df['structure_area_m2'] + df['tree_area_m2'] + df['grass_area_m2'] + 1.0
df['fuel_density'] = (df['tree_area_m2'] * 1.5) / df['estimated_lot_area']

# 2. Load XGBoost (Tabular)
# We assume you just retrain it quickly here to ensure freshness
feature_cols = [c for c in df.columns if c not in ['id', 'target', 'lat', 'lon', 'address', 'filename', 'risk_factor', 'fuel_density', 'estimated_lot_area']]
# Note: The engineered features X above included interactions. 
# For this simple export, we'll just train on the raw features + simple engineered ones if we wanted.
# But to match the notebook logic, let's stick to raw features for the "Base" XGBoost or use the X we got.

# Let's just use the features available in df for simplicity of the app pipeline
train_cols = [c for c in df.columns if c in ['structure_area_m2', 'tree_area_m2', 'grass_area_m2', 'defensible_space_m', 'compactness', 'tree_count']]
X_simple = df[train_cols].values

xgb = XGBClassifier(n_estimators=100, learning_rate=0.05)
xgb.fit(X_simple, y)
df['xgb_prob'] = xgb.predict_proba(X_simple)[:, 1]

# 3. Load CNN (Visual) - Placeholder
df['cnn_prob'] = df['xgb_prob'] # Placeholder

# 4. Calculate "Risk Factors" (Explanations)
df['risk_factor'] = 'Low Risk'
# Use 'fuel_density' which we created above
median_fuel = df['fuel_density'].median()

df.loc[df['fuel_density'] > median_fuel, 'risk_factor'] = 'High Fuel Load'
df.loc[df['defensible_space_m'] < 5, 'risk_factor'] = 'Zero Defensible Space'
df.loc[(df['fuel_density'] > median_fuel) & (df['defensible_space_m'] < 5), 'risk_factor'] = 'Critical Vulnerability'

# 5. Save App Data
os.makedirs("../app", exist_ok=True)
df.to_csv("../app/app_data.csv", index=False)
print("✅ App data prepared at ../app/app_data.csv")